In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [ ]:
model_name = "cross-encoder/nli-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

id2label = model.config.id2label
label2id = {v.lower(): k for k, v in id2label.items()}
entailment_id = label2id["entailment"]

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")
print(f"Label mapping: {id2label}")
print(f"Entailment label id: {entailment_id}")

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
def predict_paraphrase_probabilities(sentence1_list, sentence2_list, batch_size=32, max_length=256):
    paraphrase_probs = []
    predicted_labels = []
    confidences = []
    raw_entailment_scores = []

    for start_idx in range(0, len(sentence1_list), batch_size):
        batch_s1 = sentence1_list[start_idx:start_idx + batch_size]
        batch_s2 = sentence2_list[start_idx:start_idx + batch_size]

        inputs = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)
            entailment_probs = probs[:, entailment_id]
            batch_predictions = (entailment_probs >= 0.5).long()
            batch_confidences = torch.abs(entailment_probs - 0.5)

        paraphrase_probs.extend(entailment_probs.detach().cpu().tolist())
        predicted_labels.extend(batch_predictions.detach().cpu().tolist())
        confidences.extend(batch_confidences.detach().cpu().tolist())
        raw_entailment_scores.extend(logits[:, entailment_id].detach().cpu().tolist())

    return paraphrase_probs, predicted_labels, confidences, raw_entailment_scores

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

paraphrase_probabilities, predictions, confidences, raw_entailment_scores = predict_paraphrase_probabilities(
    sentence1_list,
    sentence2_list,
    batch_size=32,
    max_length=256
)

print(f"Completed cross-encoder inference for {len(predictions)} examples.")
print("Decision rule: paraphrase if entailment probability >= 0.5")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_probs = [p for p, label in zip(paraphrase_probabilities, labels) if label == 1]
negative_probs = [p for p, label in zip(paraphrase_probabilities, labels) if label == 0]
mean_positive_probability = sum(positive_probs) / len(positive_probs)
mean_negative_probability = sum(negative_probs) / len(negative_probs)

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean paraphrase probability | label=1: {mean_positive_probability:.4f}")
print(f"Mean paraphrase probability | label=0: {mean_negative_probability:.4f}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
num_examples_to_show = 8

ranked_indices = sorted(range(len(dataset)), key=lambda i: confidences[i], reverse=True)

for rank, i in enumerate(ranked_indices[:num_examples_to_show], start=1):
    row = dataset[i]
    true_label = labels[i]
    pred_label = predictions[i]
    prob = paraphrase_probabilities[i]
    confidence = confidences[i]
    entailment_logit = raw_entailment_scores[i]
    print(f"High-confidence example {rank}")
    print(f"index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {true_label} ({label_map[true_label]})")
    print(f"pred label: {pred_label} ({label_map[pred_label]})")
    print(f"paraphrase probability: {prob:.4f}")
    print(f"prediction confidence: {confidence:.4f}")
    print(f"entailment logit: {entailment_logit:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_method=cross_encoder_sentence_pair_entailment_probability")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print("decision_threshold=0.5")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_paraphrase_probability_label1={mean_positive_probability:.4f}")
print(f"mean_paraphrase_probability_label0={mean_negative_probability:.4f}")